In [8]:
# ============================================================
# WALK-FORWARD CV — RF benchmark + XGB classifier + XGB profit regressor
# Based on your notebook structure, but upgraded for:
#   1) F1 optimization
#   2) profit magnitude ranking
# ============================================================

import gc
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)
from xgboost import XGBClassifier, XGBRegressor

In [9]:
DATA_PATH = r"D:/Data Challenge/master_dataset.parquet"

df = pd.read_parquet(DATA_PATH).copy()

print("Shape:", df.shape)
print("Columns:", len(df.columns))
print("\nColumns list:")
print(df.columns.tolist())

# Ensure MONTH exists and is ordered as string YYYY-MM
assert "MONTH" in df.columns, "MONTH column not found."
assert "TARGET" in df.columns, "TARGET column not found."
assert "PROFIT" in df.columns, "PROFIT column not found."

df["MONTH"] = df["MONTH"].astype(str)
df["TARGET"] = pd.to_numeric(df["TARGET"], errors="coerce")
df["PROFIT"] = pd.to_numeric(df["PROFIT"], errors="coerce")

# Keep only rows with valid binary target
df = df[df["TARGET"].isin([0, 1])].copy()
df["TARGET"] = df["TARGET"].astype(int)

print("\nTarget distribution:")
print(df["TARGET"].value_counts(dropna=False))
print("\nPositive rate:", round(df["TARGET"].mean(), 4))

print("\nMonths:")
months_sorted = sorted(df["MONTH"].dropna().unique().tolist())
print(f"Nb months: {len(months_sorted)}")
print("First 6:", months_sorted[:6])
print("Last 6 :", months_sorted[-6:])

Shape: (211510, 71)
Columns: 71

Columns list:
['EID', 'MONTH', 'PEAKID', 'is_sim_only', 'PR', 'PR_signed', 'C', 'PROFIT', 'TARGET', 'pr_partial_current', 'DECISION_MONTH', 'has_pr_history', 'has_profit_history', 'has_cost_history', 'pr_lag1', 'pr_lag2', 'pr_lag3', 'c_lag1', 'profit_lag1', 'profit_lag2', 'target_lag1', 'pr_rolling3_mean', 'profit_rolling3_mean', 'profitable_count_3m', 'psd_nonzero_count', 'psd_abs_nonzero_mean', 'psd_abs_nonzero_std', 'psd_abs_sum', 'psd_signed_mean', 'psd_abs_max', 'activation_mean', 'activation_max', 'activation_nonzero_count', 'wind_abs_mean', 'solar_abs_mean', 'hydro_abs_mean', 'nonrenew_abs_mean', 'external_abs_mean', 'load_abs_mean', 'transoutage_abs_mean', 'hydro_log_abs_mean', 'hydro_abs_max', 'wind_log_abs_mean', 'wind_abs_max', 'load_log_abs_mean', 'load_abs_max', 'psd_abs_s1_mean', 'psd_abs_s23_mean', 'psd_scenario_spread', 'psd_abs_early', 'psd_abs_late', 'psm_nonzero_count', 'psm_abs_nonzero_mean', 'psm_abs_nonzero_std', 'psm_abs_sum', 'ps

In [10]:
# ============================================================
# 2) FEATURE LOGIC
# ============================================================

TARGET_COL = "TARGET"
PROFIT_COL = "PROFIT"
MONTH_COL = "MONTH"

# Explicitly removed
DROP_COLS = [
    "EID",              # pure identifier
    "MONTH",            # used only for walk-forward split
    "DECISION_MONTH",   # risky / redundant time index
    "PR",               # direct realized outcome information
    "PR_signed",
    "C",
    "PROFIT",
    "TARGET",
    "season",           # redundant with month/year
    "season_encoded",
    "is_sim_only"
]

drop_cols_existing = [c for c in DROP_COLS if c in df.columns]

# Keep only remaining numeric / bool columns
candidate_cols = [c for c in df.columns if c not in drop_cols_existing]
feature_cols = df[candidate_cols].select_dtypes(include=[np.number, "bool"]).columns.tolist()

print("\nDropped columns:")
print(drop_cols_existing)

print("\nFeature count:", len(feature_cols))
print("Feature columns:")
print(feature_cols)


Dropped columns:
['EID', 'MONTH', 'DECISION_MONTH', 'PR', 'PR_signed', 'C', 'PROFIT', 'TARGET', 'season', 'season_encoded', 'is_sim_only']

Feature count: 60
Feature columns:
['PEAKID', 'pr_partial_current', 'has_pr_history', 'has_profit_history', 'has_cost_history', 'pr_lag1', 'pr_lag2', 'pr_lag3', 'c_lag1', 'profit_lag1', 'profit_lag2', 'target_lag1', 'pr_rolling3_mean', 'profit_rolling3_mean', 'profitable_count_3m', 'psd_nonzero_count', 'psd_abs_nonzero_mean', 'psd_abs_nonzero_std', 'psd_abs_sum', 'psd_signed_mean', 'psd_abs_max', 'activation_mean', 'activation_max', 'activation_nonzero_count', 'wind_abs_mean', 'solar_abs_mean', 'hydro_abs_mean', 'nonrenew_abs_mean', 'external_abs_mean', 'load_abs_mean', 'transoutage_abs_mean', 'hydro_log_abs_mean', 'hydro_abs_max', 'wind_log_abs_mean', 'wind_abs_max', 'load_log_abs_mean', 'load_abs_max', 'psd_abs_s1_mean', 'psd_abs_s23_mean', 'psd_scenario_spread', 'psd_abs_early', 'psd_abs_late', 'psm_nonzero_count', 'psm_abs_nonzero_mean', 'psm_

In [12]:
def make_walkforward_splits(months, min_train_months=6, valid_window=1):
    """
    Expanding-window walk-forward CV.

    Example:
      months = [m1,m2,m3,m4,m5,m6,m7]
      min_train_months=4, valid_window=1

      Fold 1: train m1..m4, valid m5
      Fold 2: train m1..m5, valid m6
      Fold 3: train m1..m6, valid m7
    """
    splits = []

    for i in range(min_train_months, len(months) - valid_window + 1):
        train_months = months[:i]
        valid_months = months[i:i + valid_window]
        splits.append((train_months, valid_months))

    return splits

splits = splits = make_walkforward_splits(
    sorted(df[MONTH_COL].dropna().astype(str).unique().tolist()),
    min_train_months=6,
    valid_window=1
)

print("Nb folds:", len(splits))
for i, (tr, va) in enumerate(splits[:5], 1):
    print(f"Fold {i}: train {tr[0]}->{tr[-1]} | valid {va[0]}->{va[-1]}")


Nb folds: 42
Fold 1: train 2020-01->2020-06 | valid 2020-07->2020-07
Fold 2: train 2020-01->2020-07 | valid 2020-08->2020-08
Fold 3: train 2020-01->2020-08 | valid 2020-09->2020-09
Fold 4: train 2020-01->2020-09 | valid 2020-10->2020-10
Fold 5: train 2020-01->2020-10 | valid 2020-11->2020-11


In [13]:
# ============================================================
# 4) HELPERS
# ============================================================

def find_best_threshold(y_true, y_prob, thresholds=np.arange(0.01, 0.991, 0.01)):
    rows = []
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        rows.append({
            "threshold": t,
            "f1": f1_score(y_true, pred, zero_division=0),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "n_pred_pos": int(pred.sum())
        })
    tab = pd.DataFrame(rows)
    best = tab.loc[tab["f1"].idxmax()].to_dict()
    return best, tab

def topk_summary(y_true, score, realized_profit=None, k=100):
    tmp = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(score)
    })

    if realized_profit is not None:
        tmp["realized_profit"] = np.asarray(realized_profit)

    tmp = tmp.sort_values("score", ascending=False).head(k)

    if len(tmp) == 0:
        out = {
            f"top{k}_precision": np.nan,
            f"top{k}_positives": 0,
        }
        if realized_profit is not None:
            out[f"top{k}_profit"] = 0.0
        return out

    out = {
        f"top{k}_precision": float(tmp["y_true"].mean()),
        f"top{k}_positives": int(tmp["y_true"].sum()),
    }

    if realized_profit is not None:
        out[f"top{k}_profit"] = float(tmp["realized_profit"].sum())

    return out

def summarize_selected_profit(df_eval, pred_col, profit_col="PROFIT", target_col="TARGET"):
    selected = df_eval[df_eval[pred_col] == 1].copy()

    if len(selected) == 0:
        return {
            "n_selected": 0,
            "selected_positive_rate": np.nan,
            "total_realized_profit_selected": 0.0,
            "avg_realized_profit_selected": np.nan
        }

    return {
        "n_selected": int(len(selected)),
        "selected_positive_rate": float(selected[target_col].mean()),
        "total_realized_profit_selected": float(selected[profit_col].sum()),
        "avg_realized_profit_selected": float(selected[profit_col].mean())
    }

In [14]:
# ============================================================
# 5) MAIN WALK-FORWARD FUNCTION
# ============================================================

def run_walkforward_cv_dual_objective(
    df,
    feature_cols,
    target_col="TARGET",
    month_col="MONTH",
    profit_col="PROFIT",
    min_train_months=6,
    valid_window=1,
    topk_list=(50, 100, 250, 500),
    rf_params=None,
    xgb_clf_params=None,
    xgb_reg_params=None,
):
    if rf_params is None:
        rf_params = {
            "n_estimators": 300,
            "max_depth": 8,
            "min_samples_leaf": 20,
            "class_weight": "balanced",
            "random_state": 42,
            "n_jobs": -1
        }

    if xgb_clf_params is None:
        xgb_clf_params = {
            "n_estimators": 400,
            "max_depth": 6,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.0,
            "min_child_weight": 3,
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "random_state": 42,
            "n_jobs": -1
        }

    if xgb_reg_params is None:
        xgb_reg_params = {
            "n_estimators": 300,
            "max_depth": 5,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.0,
            "min_child_weight": 3,
            "objective": "reg:squarederror",
            "random_state": 42,
            "n_jobs": -1
        }

    months = sorted(df[month_col].dropna().astype(str).unique().tolist())
    splits = make_walkforward_splits(
        months=months,
        min_train_months=min_train_months,
        valid_window=valid_window
    )

    print(f"\nTotal folds: {len(splits)}")

    cv_rows = []
    oof_parts = []
    feat_imp_parts = []

    for fold_id, (train_months, valid_months) in enumerate(splits, start=1):
        train_mask = df[month_col].isin(train_months)
        valid_mask = df[month_col].isin(valid_months)

        train_df = df.loc[train_mask].copy()
        valid_df = df.loc[valid_mask].copy()

        X_train = train_df[feature_cols].copy().fillna(0)
        y_train = train_df[target_col].astype(int)
        p_train = pd.to_numeric(train_df[profit_col], errors="coerce")

        X_valid = valid_df[feature_cols].copy().fillna(0)
        y_valid = valid_df[target_col].astype(int)
        p_valid = pd.to_numeric(valid_df[profit_col], errors="coerce")

        # Safety checks
        assert set(train_months).isdisjoint(set(valid_months)), "Train/valid overlap detected."
        assert train_df[month_col].max() < valid_df[month_col].min(), "Temporal ordering broken."

        # Imbalance ratio for XGB
        n_pos = int((y_train == 1).sum())
        n_neg = int((y_train == 0).sum())
        scale_pos_weight = n_neg / max(n_pos, 1)

        # -----------------------------
        # Random Forest classifier
        # -----------------------------
        rf_model = RandomForestClassifier(**rf_params)
        rf_model.fit(X_train, y_train)
        rf_prob = rf_model.predict_proba(X_valid)[:, 1]

        # -----------------------------
        # XGBoost classifier
        # -----------------------------
        xgb_params_fold = xgb_clf_params.copy()
        xgb_params_fold["scale_pos_weight"] = scale_pos_weight

        xgb_clf = XGBClassifier(**xgb_params_fold)
        xgb_clf.fit(X_train, y_train)
        xgb_prob = xgb_clf.predict_proba(X_valid)[:, 1]

        # -----------------------------
        # XGBoost regressor on positive rows only
        # -----------------------------
        pos_mask = (y_train == 1) & p_train.notna()
        X_train_pos = X_train.loc[pos_mask].copy()
        p_train_pos = p_train.loc[pos_mask].copy()

        if len(X_train_pos) >= 20:
            xgb_reg = XGBRegressor(**xgb_reg_params)
            xgb_reg.fit(X_train_pos, p_train_pos)
            pred_profit = xgb_reg.predict(X_valid)
            pred_profit = np.maximum(pred_profit, 0)
        else:
            fallback_val = float(np.nanmean(p_train_pos)) if len(p_train_pos) > 0 else 0.0
            pred_profit = np.repeat(max(fallback_val, 0.0), len(X_valid))

        # EV scores
        rf_ev_score = rf_prob * pred_profit
        xgb_ev_score = xgb_prob * pred_profit

        # Fold metrics
        row = {
            "fold": fold_id,
            "train_start": train_months[0],
            "train_end": train_months[-1],
            "valid_start": valid_months[0],
            "valid_end": valid_months[-1],
            "n_train": len(train_df),
            "n_valid": len(valid_df),
            "train_pos_rate": float(y_train.mean()),
            "valid_pos_rate": float(y_valid.mean()),
            "scale_pos_weight": float(scale_pos_weight),

            "rf_roc_auc": roc_auc_score(y_valid, rf_prob) if y_valid.nunique() > 1 else np.nan,
            "rf_pr_auc": average_precision_score(y_valid, rf_prob) if y_valid.nunique() > 1 else np.nan,

            "xgb_roc_auc": roc_auc_score(y_valid, xgb_prob) if y_valid.nunique() > 1 else np.nan,
            "xgb_pr_auc": average_precision_score(y_valid, xgb_prob) if y_valid.nunique() > 1 else np.nan,
        }

        for k in topk_list:
            row.update({f"rf_{kk}": vv for kk, vv in topk_summary(y_valid, rf_prob, p_valid, k=k).items()})
            row.update({f"xgb_{kk}": vv for kk, vv in topk_summary(y_valid, xgb_prob, p_valid, k=k).items()})
            row.update({f"xgb_ev_{kk}": vv for kk, vv in topk_summary(y_valid, xgb_ev_score, p_valid, k=k).items()})

        cv_rows.append(row)

        # OOF predictions
        oof_fold = valid_df.copy()
        oof_fold["cv_fold"] = fold_id
        oof_fold["rf_prob"] = rf_prob
        oof_fold["xgb_prob"] = xgb_prob
        oof_fold["pred_profit"] = pred_profit
        oof_fold["rf_ev_score"] = rf_ev_score
        oof_fold["xgb_ev_score"] = xgb_ev_score

        oof_parts.append(oof_fold)

        # Feature importances
        feat_imp_parts.append(pd.DataFrame({
            "feature": feature_cols,
            "importance": rf_model.feature_importances_,
            "model": "rf",
            "fold": fold_id
        }))

        feat_imp_parts.append(pd.DataFrame({
            "feature": feature_cols,
            "importance": xgb_clf.feature_importances_,
            "model": "xgb_clf",
            "fold": fold_id
        }))

        print(
            f"Fold {fold_id:02d} | "
            f"train {train_months[0]} -> {train_months[-1]} | "
            f"valid {valid_months[0]} -> {valid_months[-1]} | "
            f"RF PR-AUC={row['rf_pr_auc']:.4f} | "
            f"XGB PR-AUC={row['xgb_pr_auc']:.4f} | "
            f"scale_pos_weight={scale_pos_weight:.2f}"
        )

        del train_df, valid_df, X_train, y_train, p_train, X_valid, y_valid, p_valid
        del rf_model, xgb_clf, rf_prob, xgb_prob, pred_profit, rf_ev_score, xgb_ev_score
        gc.collect()

    cv_results_df = pd.DataFrame(cv_rows)
    oof_df = pd.concat(oof_parts, ignore_index=True)
    feat_imp_df = pd.concat(feat_imp_parts, ignore_index=True)

    return cv_results_df, oof_df, feat_imp_df

In [15]:
# ============================================================
# 6) PARAMETERS
# ============================================================

rf_params = {
    "n_estimators": 300,
    "max_depth": 8,
    "min_samples_leaf": 20,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1
}

xgb_clf_params = {
    "n_estimators": 400,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "min_child_weight": 3,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}

xgb_reg_params = {
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "min_child_weight": 3,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1
}

In [16]:
# ============================================================
# 7) RUN WALK-FORWARD CV
# ============================================================

cv_results_df, oof_df, feat_imp_df = run_walkforward_cv_dual_objective(
    df=df,
    feature_cols=feature_cols,
    target_col=TARGET_COL,
    month_col=MONTH_COL,
    profit_col=PROFIT_COL,
    min_train_months=6,
    valid_window=1,
    topk_list=(50, 100, 250, 500),
    rf_params=rf_params,
    xgb_clf_params=xgb_clf_params,
    xgb_reg_params=xgb_reg_params
)



Total folds: 42
Fold 01 | train 2020-01 -> 2020-06 | valid 2020-07 -> 2020-07 | RF PR-AUC=0.3663 | XGB PR-AUC=0.3817 | scale_pos_weight=8.70
Fold 02 | train 2020-01 -> 2020-07 | valid 2020-08 -> 2020-08 | RF PR-AUC=0.4065 | XGB PR-AUC=0.3572 | scale_pos_weight=9.19
Fold 03 | train 2020-01 -> 2020-08 | valid 2020-09 -> 2020-09 | RF PR-AUC=0.4762 | XGB PR-AUC=0.4392 | scale_pos_weight=9.45
Fold 04 | train 2020-01 -> 2020-09 | valid 2020-10 -> 2020-10 | RF PR-AUC=0.4118 | XGB PR-AUC=0.3905 | scale_pos_weight=9.27
Fold 05 | train 2020-01 -> 2020-10 | valid 2020-11 -> 2020-11 | RF PR-AUC=0.4392 | XGB PR-AUC=0.4206 | scale_pos_weight=9.06
Fold 06 | train 2020-01 -> 2020-11 | valid 2020-12 -> 2020-12 | RF PR-AUC=0.3981 | XGB PR-AUC=0.4102 | scale_pos_weight=8.79
Fold 07 | train 2020-01 -> 2020-12 | valid 2021-01 -> 2021-01 | RF PR-AUC=0.4313 | XGB PR-AUC=0.3946 | scale_pos_weight=8.83
Fold 08 | train 2020-01 -> 2021-01 | valid 2021-02 -> 2021-02 | RF PR-AUC=0.4085 | XGB PR-AUC=0.4001 | scale

In [17]:
# ============================================================
# 8) FOLD RESULTS SUMMARY
# ============================================================

print("\n================ FOLD RESULTS ================\n")
print(cv_results_df.head())

metric_cols = [c for c in cv_results_df.columns if any(x in c for x in ["auc", "precision", "positives", "profit"])]

print("\nMean fold metrics:")
print(cv_results_df[metric_cols].mean(numeric_only=True).to_frame("mean").T)

print("\nStd fold metrics:")
print(cv_results_df[metric_cols].std(numeric_only=True).to_frame("std").T)


================ FOLD RESULTS ================

   fold train_start train_end valid_start valid_end  n_train  n_valid  \
0     1     2020-01   2020-06     2020-07   2020-07    14047     2636   
1     2     2020-01   2020-07     2020-08   2020-08    16683     2636   
2     3     2020-01   2020-08     2020-09   2020-09    19319     2636   
3     4     2020-01   2020-09     2020-10   2020-10    21955     2636   
4     5     2020-01   2020-10     2020-11   2020-11    24591     2636   

   train_pos_rate  valid_pos_rate  scale_pos_weight  ...  \
0        0.103083        0.071700          8.700967  ...   
1        0.098124        0.080046          9.191203  ...   
2        0.095657        0.110015          9.454004  ...   
3        0.097381        0.116085          9.268943  ...   
4        0.099386        0.127845          9.061784  ...   

   xgb_ev_top250_profit  rf_top500_precision  rf_top500_positives  \
0          85602.328268                0.228                  114   
1          22

In [18]:
# ============================================================
# 9) GLOBAL OOF THRESHOLD SELECTION FOR F1
# ============================================================

rf_best, rf_thresh_tab = find_best_threshold(oof_df[TARGET_COL].values, oof_df["rf_prob"].values)
xgb_best, xgb_thresh_tab = find_best_threshold(oof_df[TARGET_COL].values, oof_df["xgb_prob"].values)

print("\nBest RF threshold from OOF:")
print(rf_best)

print("\nBest XGB threshold from OOF:")
print(xgb_best)

rf_best_t = float(rf_best["threshold"])
xgb_best_t = float(xgb_best["threshold"])

oof_df["rf_pred"] = (oof_df["rf_prob"] >= rf_best_t).astype(int)
oof_df["xgb_pred"] = (oof_df["xgb_prob"] >= xgb_best_t).astype(int)


Best RF threshold from OOF:
{'threshold': 0.67, 'f1': 0.4349113378173977, 'precision': 0.39479988314344144, 'recall': 0.48409514257056885, 'n_pred_pos': 17115.0}

Best XGB threshold from OOF:
{'threshold': 0.66, 'f1': 0.4259969187775429, 'precision': 0.42607324589693973, 'recall': 0.4259206189998567, 'n_pred_pos': 13953.0}


In [19]:
# ============================================================
# 10) OOF CLASSIFICATION SUMMARY
# ============================================================

oof_classif_summary = pd.DataFrame([
    {
        "model": "RandomForest",
        "threshold": rf_best_t,
        "f1": f1_score(oof_df[TARGET_COL], oof_df["rf_pred"], zero_division=0),
        "precision": precision_score(oof_df[TARGET_COL], oof_df["rf_pred"], zero_division=0),
        "recall": recall_score(oof_df[TARGET_COL], oof_df["rf_pred"], zero_division=0),
        "roc_auc": roc_auc_score(oof_df[TARGET_COL], oof_df["rf_prob"]),
        "pr_auc": average_precision_score(oof_df[TARGET_COL], oof_df["rf_prob"]),
    },
    {
        "model": "XGBoost",
        "threshold": xgb_best_t,
        "f1": f1_score(oof_df[TARGET_COL], oof_df["xgb_pred"], zero_division=0),
        "precision": precision_score(oof_df[TARGET_COL], oof_df["xgb_pred"], zero_division=0),
        "recall": recall_score(oof_df[TARGET_COL], oof_df["xgb_pred"], zero_division=0),
        "roc_auc": roc_auc_score(oof_df[TARGET_COL], oof_df["xgb_prob"]),
        "pr_auc": average_precision_score(oof_df[TARGET_COL], oof_df["xgb_prob"]),
    }
]).sort_values("f1", ascending=False)

print("\n================ OOF CLASSIFICATION SUMMARY ================\n")
print(oof_classif_summary)

print("\nConfusion matrix — XGB")
print(confusion_matrix(oof_df[TARGET_COL], oof_df["xgb_pred"]))

print("\nClassification report — XGB")
print(classification_report(oof_df[TARGET_COL], oof_df["xgb_pred"], zero_division=0))



================ OOF CLASSIFICATION SUMMARY ================

          model  threshold        f1  precision    recall   roc_auc    pr_auc
0  RandomForest       0.67  0.434911   0.394800  0.484095  0.841312  0.400249
1       XGBoost       0.66  0.425997   0.426073  0.425921  0.827384  0.391709

Confusion matrix — XGB
[[175497   8008]
 [  8013   5945]]

Classification report — XGB
              precision    recall  f1-score   support

           0       0.96      0.96      0.96    183505
           1       0.43      0.43      0.43     13958

    accuracy                           0.92    197463
   macro avg       0.69      0.69      0.69    197463
weighted avg       0.92      0.92      0.92    197463



In [20]:
# ============================================================
# 11) OOF PROFIT OF THRESHOLD-SELECTED CASES
# ============================================================

rf_profit_summary = summarize_selected_profit(oof_df, "rf_pred", profit_col=PROFIT_COL, target_col=TARGET_COL)
xgb_profit_summary = summarize_selected_profit(oof_df, "xgb_pred", profit_col=PROFIT_COL, target_col=TARGET_COL)

profit_compare = pd.DataFrame([
    {"model": "RandomForest", **rf_profit_summary},
    {"model": "XGBoost", **xgb_profit_summary},
])

print("\n================ PROFIT OF THRESHOLD-SELECTED CASES ================\n")
print(profit_compare)


================ PROFIT OF THRESHOLD-SELECTED CASES ================

          model  n_selected  selected_positive_rate  \
0  RandomForest       17115                0.394800   
1       XGBoost       13953                0.426073   

   total_realized_profit_selected  avg_realized_profit_selected  
0                    1.146231e+07                    669.723188  
1                    1.060978e+07                    760.394310  


In [21]:
# ============================================================
# 11) OOF PROFIT OF THRESHOLD-SELECTED CASES
# ============================================================

rf_profit_summary = summarize_selected_profit(oof_df, "rf_pred", profit_col=PROFIT_COL, target_col=TARGET_COL)
xgb_profit_summary = summarize_selected_profit(oof_df, "xgb_pred", profit_col=PROFIT_COL, target_col=TARGET_COL)

profit_compare = pd.DataFrame([
    {"model": "RandomForest", **rf_profit_summary},
    {"model": "XGBoost", **xgb_profit_summary},
])

print("\n================ PROFIT OF THRESHOLD-SELECTED CASES ================\n")
print(profit_compare)


================ PROFIT OF THRESHOLD-SELECTED CASES ================

          model  n_selected  selected_positive_rate  \
0  RandomForest       17115                0.394800   
1       XGBoost       13953                0.426073   

   total_realized_profit_selected  avg_realized_profit_selected  
0                    1.146231e+07                    669.723188  
1                    1.060978e+07                    760.394310  


In [22]:
# ============================================================
# 12) TOP-K BUSINESS EVALUATION
# ============================================================

def evaluate_topk_strategies(oof_df, k_list=(50, 100, 250, 500)):
    rows = []

    for k in k_list:
        for strategy, score_col in [
            ("RF probability", "rf_prob"),
            ("XGB probability", "xgb_prob"),
            ("RF EV score", "rf_ev_score"),
            ("XGB EV score", "xgb_ev_score"),
        ]:
            topk = oof_df.sort_values(score_col, ascending=False).head(k)

            rows.append({
                "strategy": strategy,
                "k": k,
                "positives_in_topk": int(topk[TARGET_COL].sum()),
                "precision_topk": float(topk[TARGET_COL].mean()),
                "realized_profit_topk": float(topk[PROFIT_COL].sum()),
                "avg_profit_topk": float(topk[PROFIT_COL].mean())
            })

    return pd.DataFrame(rows)

topk_eval_df = evaluate_topk_strategies(oof_df, k_list=(50, 100, 250, 500))

print("\n================ TOP-K BUSINESS EVALUATION ================\n")
print(topk_eval_df.sort_values(["k", "realized_profit_topk"], ascending=[True, False]))


================ TOP-K BUSINESS EVALUATION ================

           strategy    k  positives_in_topk  precision_topk  \
3      XGB EV score   50                 24           0.480   
2       RF EV score   50                 23           0.460   
0    RF probability   50                 44           0.880   
1   XGB probability   50                 47           0.940   
7      XGB EV score  100                 51           0.510   
4    RF probability  100                 87           0.870   
5   XGB probability  100                 89           0.890   
6       RF EV score  100                 42           0.420   
11     XGB EV score  250                119           0.476   
8    RF probability  250                221           0.884   
10      RF EV score  250                114           0.456   
9   XGB probability  250                217           0.868   
15     XGB EV score  500                237           0.474   
14      RF EV score  500                224           0.

In [23]:
# ============================================================
# NEXT STEP — SMALL RF HYPERPARAMETER SEARCH WITH WALK-FORWARD CV
# Goal: improve the current RF leader on F1
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score
)
import pandas as pd
import numpy as np
import gc

# ------------------------------------------------------------
# 1) Small RF grid
# ------------------------------------------------------------
rf_grid = [
    {"n_estimators": 300, "max_depth": 6,   "min_samples_leaf": 10, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 8,   "min_samples_leaf": 10, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 10,  "min_samples_leaf": 10, "max_features": "sqrt"},
    {"n_estimators": 500, "max_depth": 8,   "min_samples_leaf": 10, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 8,   "min_samples_leaf": 20, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 8,   "min_samples_leaf": 40, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": None,"min_samples_leaf": 20, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 8,   "min_samples_leaf": 20, "max_features": 0.5},
]

# ------------------------------------------------------------
# 2) Reusable RF walk-forward runner
# ------------------------------------------------------------
def run_walkforward_rf_only(
    df,
    feature_cols,
    target_col="TARGET",
    month_col="MONTH",
    profit_col="PROFIT",
    min_train_months=6,
    valid_window=1,
    rf_params=None,
):
    months = sorted(df[month_col].dropna().astype(str).unique().tolist())
    splits = make_walkforward_splits(
        months=months,
        min_train_months=min_train_months,
        valid_window=valid_window
    )

    oof_parts = []
    fold_rows = []

    for fold_id, (train_months, valid_months) in enumerate(splits, start=1):
        train_df = df[df[month_col].isin(train_months)].copy()
        valid_df = df[df[month_col].isin(valid_months)].copy()

        X_train = train_df[feature_cols].fillna(0)
        y_train = train_df[target_col].astype(int)

        X_valid = valid_df[feature_cols].fillna(0)
        y_valid = valid_df[target_col].astype(int)
        p_valid = pd.to_numeric(valid_df[profit_col], errors="coerce")

        model = RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            **rf_params
        )
        model.fit(X_train, y_train)
        rf_prob = model.predict_proba(X_valid)[:, 1]

        fold_rows.append({
            "fold": fold_id,
            "train_start": train_months[0],
            "train_end": train_months[-1],
            "valid_start": valid_months[0],
            "valid_end": valid_months[-1],
            "rf_pr_auc": average_precision_score(y_valid, rf_prob) if y_valid.nunique() > 1 else np.nan,
            "rf_roc_auc": roc_auc_score(y_valid, rf_prob) if y_valid.nunique() > 1 else np.nan,
        })

        part = valid_df[[target_col, profit_col, month_col]].copy()
        part["rf_prob"] = rf_prob
        oof_parts.append(part)

        del train_df, valid_df, X_train, y_train, X_valid, y_valid, p_valid, model, rf_prob
        gc.collect()

    oof_df_rf = pd.concat(oof_parts, ignore_index=True)
    fold_df_rf = pd.DataFrame(fold_rows)

    # OOF threshold tuning
    rf_best, rf_thresh_tab = find_best_threshold(
        oof_df_rf[target_col].values,
        oof_df_rf["rf_prob"].values
    )
    rf_best_t = float(rf_best["threshold"])
    oof_df_rf["rf_pred"] = (oof_df_rf["rf_prob"] >= rf_best_t).astype(int)

    # OOF metrics
    summary = {
        "threshold": rf_best_t,
        "f1": f1_score(oof_df_rf[target_col], oof_df_rf["rf_pred"], zero_division=0),
        "precision": precision_score(oof_df_rf[target_col], oof_df_rf["rf_pred"], zero_division=0),
        "recall": recall_score(oof_df_rf[target_col], oof_df_rf["rf_pred"], zero_division=0),
        "roc_auc": roc_auc_score(oof_df_rf[target_col], oof_df_rf["rf_prob"]),
        "pr_auc": average_precision_score(oof_df_rf[target_col], oof_df_rf["rf_prob"]),
        "n_selected": int(oof_df_rf["rf_pred"].sum()),
        "total_realized_profit_selected": float(oof_df_rf.loc[oof_df_rf["rf_pred"] == 1, profit_col].sum()),
        "avg_realized_profit_selected": float(oof_df_rf.loc[oof_df_rf["rf_pred"] == 1, profit_col].mean()) if (oof_df_rf["rf_pred"] == 1).any() else np.nan,
    }

    return summary, oof_df_rf, fold_df_rf, rf_thresh_tab

# ------------------------------------------------------------
# 3) Run tuning loop
# ------------------------------------------------------------
rf_tuning_rows = []
rf_tuning_oof_store = {}

for i, params in enumerate(rf_grid, start=1):
    print(f"\nRunning RF config {i}/{len(rf_grid)}: {params}")

    summary, oof_df_rf, fold_df_rf, rf_thresh_tab = run_walkforward_rf_only(
        df=df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        month_col=MONTH_COL,
        profit_col=PROFIT_COL,
        min_train_months=6,
        valid_window=1,
        rf_params=params
    )

    row = {
        "config_id": i,
        **params,
        **summary
    }
    rf_tuning_rows.append(row)
    rf_tuning_oof_store[i] = oof_df_rf

rf_tuning_results = pd.DataFrame(rf_tuning_rows).sort_values(
    ["f1", "pr_auc", "total_realized_profit_selected"],
    ascending=[False, False, False]
)

print("\n================ RF TUNING RESULTS ================\n")
print(rf_tuning_results)


Running RF config 1/8: {'n_estimators': 300, 'max_depth': 6, 'min_samples_leaf': 10, 'max_features': 'sqrt'}

Running RF config 2/8: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 10, 'max_features': 'sqrt'}

Running RF config 3/8: {'n_estimators': 300, 'max_depth': 10, 'min_samples_leaf': 10, 'max_features': 'sqrt'}

Running RF config 4/8: {'n_estimators': 500, 'max_depth': 8, 'min_samples_leaf': 10, 'max_features': 'sqrt'}

Running RF config 5/8: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'max_features': 'sqrt'}

Running RF config 6/8: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 40, 'max_features': 'sqrt'}

Running RF config 7/8: {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 20, 'max_features': 'sqrt'}

Running RF config 8/8: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'max_features': 0.5}

================ RF TUNING RESULTS ================

   config_id  n_estimators  max_depth  min_samples_leaf max_featu